# Phase 2: Exploratory Data Analysis & Degradation Physics (NASA C-MAPSS FD001)

**Course:** ES 335: Machine Learning (2026)  
**Scope:** Strictly restricted to the **80 Development Engines** (`16,340` operating cycles). The 20 held-out test engines remain sealed until Phase 5.

---

### Key Questions Investigated in Phase 2
1. **Sensor & Setting Screening:** Which of the 3 operating settings and 21 sensors carry real physical degradation signal vs. zero/near-zero variance?
2. **The Fixed-Schedule Dilemma:** Why does a fixed maintenance interval ($K$ cycles) inevitably force a costly trade-off between catastrophic in-service failures and wasted remaining engine life?
3. **Forward Time vs. Failure-Aligned Trajectories:** Why does aligning engines backward by *cycles before failure* ($t - T_{\text{fail}}$) collapse heterogeneous engines onto a universal degradation trajectory starting $\sim 125$ cycles prior to failure?
4. **Multivariate Correlation & PCA Health Index (Syllabus Topic 8):** Can an unsupervised 1D Principal Component Health Index capture multi-sensor compressor degradation without using RUL labels?

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(REPO_ROOT))

from src.data import load_development_data
from src.eda import (
    screen_sensors_and_settings,
    analyze_fixed_schedule_tradeoff,
    fit_pca_health_index,
)

dev_df = load_development_data(rul_cap=125, alarm_w=30)
print(f'Loaded {dev_df["unit_id"].nunique()} development engines ({len(dev_df):,} total operating cycles).')

## 1. Sensor & Operating Setting Variance Screening
We inspect standard deviation, dynamic range, and Pearson correlation with capped and uncapped RUL across all 24 telemetry channels on the 80 development engines.

In [ ]:
sensor_summary_df, retained_sensors, dropped_channels = screen_sensors_and_settings(dev_df)
print('Retained Informative Sensors (14):', retained_sensors)
print('Dropped Constant/Uninformative Channels (10):', dropped_channels)
sensor_summary_df[['channel', 'physical_description', 'mean', 'std', 'range', 'corr_rul_capped', 'status']]

## 2. Fleet Lifetime Distribution & Fixed-Schedule Replacement Trade-Off
Development engine lifetimes span from **128 to 362 cycles** (median = 199.0, mean = 204.25, std = 45.63). A static replacement schedule at cycle $K$ must either replace at $K=128$ (wasting 37.3% of fleet life) or accept high failure rates.

In [ ]:
tradeoff_df = analyze_fixed_schedule_tradeoff(dev_df)
display(tradeoff_df[tradeoff_df['replacement_cycle_K'].isin([128, 140, 160, 180, 199, 220, 250])])
display(Image(filename=str(REPO_ROOT / 'results/figures/engine_lifetime_distribution.png')))

## 3. Degradation Trajectory Alignment ($t$ vs. $-\text{RUL}$)
Aligning trajectories backward from failure ($t - T_{\text{failure}}$) demonstrates why **piecewise-linear RUL capping at 125 cycles** reflects physical reality: sensor drift is flat earlier than 125 cycles before failure, then accelerates non-linearly into the $w=30$ alarm window.

In [ ]:
display(Image(filename=str(REPO_ROOT / 'results/figures/sensor_trajectories_aligned.png')))
display(Image(filename=str(REPO_ROOT / 'results/figures/all_14_retained_sensors_trends.png')))

## 4. Sensor Correlation Structure & Unsupervised PCA Health Index (Syllabus Topic 8)
Fitting a 1D Principal Component Analysis (PCA) model on the standardized 14 retained sensors extracts a single composite health index that explains **>65% of total multi-sensor variance** and correlates strongly with RUL without using target labels.

In [ ]:
pca_df, pca_info = fit_pca_health_index(dev_df, retained_sensors)
print('PCA Summary:', pca_info)
display(Image(filename=str(REPO_ROOT / 'results/figures/sensor_correlation_and_pca_health_index.png')))